### Imports and setting paths/constants

In [13]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Patch, Rectangle
from matplotlib.backends.backend_pdf import PdfPages
from PIL import Image
import textwrap
from pathlib import Path

# Tool display names to be consistent

TOOL_DISPLAY_NAMES = {
    'jacusa2': 'JACUSA2',
    'drummer': 'DRUMMER',
    'eligos': 'ELIGOS2',
    'tombo': 'Tombo',
    'yanocomp': 'Yanocomp',
    'nanocompore': 'Nanocompore',
    'xpore': 'xPore',
    'epinano': 'EpiNano',
    'nanodoc': 'nanoDoc',
    'differr': 'DiffErr',
}

EXPECTED_COVERAGES = [
    "5x", "10x", "15x", "20x", "25x", "30x", "35x", "40x", "45x", "50x",
    "55x", "60x", "65x", "70x", "75x", "80x", "85x", "90x", "95x", "100x",
    "150x", "200x", "500x", "750x", "1000x",
]

REFERENCE_DISPLAY = {
    "16s_88_rrsE": "16S",
    "23s_78_rrlB": "23S",
}

HIGHLIGHT_TOOLS = {"drummer", "xpore", "yanocomp"}   # used for dashed boxes

# Paths

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'rnamod-tool-benchmark-code':
    PROJECT_ROOT = PROJECT_ROOT.parent

NOTEBOOK_DIR = PROJECT_ROOT / 'rnamod-tool-benchmark-code'
if not NOTEBOOK_DIR.exists():
    raise FileNotFoundError(
        f'Could not locate the rnamod-tool-benchmark-code directory from {Path.cwd().resolve()}.'
    )

FIG_DIR = NOTEBOOK_DIR / 'completeness_inflation_figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)


completeness_long_path = NOTEBOOK_DIR / "coverage_completeness_long.csv"
scope_summary_path     = NOTEBOOK_DIR / "scope_metric_summary.csv"

### Load Data

In [14]:
df_long = pd.read_csv(completeness_long_path)
df_scope = pd.read_csv(scope_summary_path)

# Map tool names everywhere
df_long["tool_display"] = df_long["tool"].map(TOOL_DISPLAY_NAMES).fillna(df_long["tool"])
df_scope["tool_display"] = df_scope["tool"].map(TOOL_DISPLAY_NAMES).fillna(df_scope["tool"])

print("Data loaded. Tools now displayed as:", sorted(df_long["tool_display"].unique()))

Data loaded. Tools now displayed as: ['DRUMMER', 'DiffErr', 'ELIGOS2', 'EpiNano', 'JACUSA2', 'Nanocompore', 'Tombo', 'Yanocomp', 'nanoDoc', 'xPore']


### Scope comparison plot - Supplementary Figure S8

In [23]:
def plot_scope_comparison(df_scope):
    bar_specs = [
        ("AUROC Universe",  "auroc_universe_mean",  "auroc_universe_std",  "#4C78A8"),
        ("AUROC Reported",  "auroc_reported_mean",  "auroc_reported_std",  "#7EB0D5"),
        ("AUPRC Universe",  "auprc_universe_mean",  "auprc_universe_std",  "#E45756"),
        ("AUPRC Reported",  "auprc_reported_mean",  "auprc_reported_std",  "#F5A898"),
    ]

    # Wider figure + a bit more height for the legend
    fig, axes = plt.subplots(2, 1, figsize=(16.5, 8.5), sharex=True)

    for ax, ref_id in zip(axes, ["16s_88_rrsE", "23s_78_rrlB"]):
        sub = df_scope[df_scope["reference"] == ref_id].copy()
        sub = sub.sort_values("tool").reset_index(drop=True)
        
        tools_disp = sub["tool_display"].tolist()
        x = np.arange(len(tools_disp))
        
        n_bars = len(bar_specs)
        bar_width = 0.18
        total_group_width = n_bars * bar_width
        # Centered offsets for the 4 bars
        offsets = [-(total_group_width / 2) + i * bar_width + bar_width/2 for i in range(n_bars)]

        for offset, (label, mean_col, std_col, color) in zip(offsets, bar_specs):
            ax.bar(x + offset, sub[mean_col], bar_width,
                   yerr=sub[std_col], capsize=2, color=color, alpha=0.92, label=label)

        ax.set_ylim(0, 1.02)
        ax.grid(axis="y", linestyle=":", alpha=0.35)
        ax.set_xticks(x)
        ax.set_xticklabels(tools_disp, rotation=0, ha="center", fontsize=14)   # slightly smaller than 12
        ax.set_ylabel("Score", fontsize=14)
        ax.set_title(f"{REFERENCE_DISPLAY[ref_id]}: Universe vs Reported "
                     "(mean ± SD across available coverages)", fontsize=14)

        # Correct dashed boxes around only the highlighted tools
        for i, t in enumerate(sub["tool"]):
            if t in HIGHLIGHT_TOOLS:
                # Tight box around the 4 bars of that tool
                ax.add_patch(Rectangle(
                    (i - total_group_width/2 - 0.05, 0),          # x start
                    total_group_width + 0.10,                     # width
                    1.02,
                    fill=False, edgecolor="#333333", lw=1.5, ls="--", zorder=5))

    # Legend (same as before)
    legend_handles = [Patch(facecolor=c, label=l) for l, _, _, c in bar_specs] + [
        Patch(facecolor="none", edgecolor="#333333", ls="--", lw=1.5,
              label="DRUMMER / xPore / yanocomp")
    ]
    fig.legend(handles=legend_handles, loc="lower center", ncol=5,
               fontsize=14, frameon=True, bbox_to_anchor=(0.5, -0.02))

    plt.tight_layout(rect=[0, 0.06, 1, 1])   # extra bottom margin for legend
    ax.set_xlim(-0.6, len(tools_disp) - 0.4)   # removes empty space on right

    # Save
    for ext in ["png", "pdf", "svg"]:
        fig.savefig(FIG_DIR / f"suppfig_S8_scope_comparison.{ext}",
                    dpi=300 if ext == "png" else None,
                    bbox_inches="tight")
    plt.close(fig)
    print("✓ suppfig_S8_scope_comparison.png (and pdf/svg) saved with nice tool names")

plot_scope_comparison(df_scope)

✓ suppfig_S8_scope_comparison.png (and pdf/svg) saved with nice tool names


### COMPLETENESS HEATMAP (Supplementary Figure S7)

In [24]:

def plot_completeness_heatmap(df_long, output_dir=RESULTS_DIR):
    cmap = plt.matplotlib.colors.LinearSegmentedColormap.from_list(
        "completeness", ["#d73027", "#fee08b", "#1a9850"], N=256)
    cmap.set_bad(color="#f0f0f0")

    fig, axes = plt.subplots(2, 1, figsize=(16, 7), gridspec_kw={"hspace": 0.45})

    for ax, ref_id in zip(axes, ["16s_88_rrsE", "23s_78_rrlB"]):
        sub = df_long[df_long["reference"] == ref_id].copy()
        pivot = sub.pivot(index="tool_display", columns="coverage_label",
                          values="completeness_pct")
        pivot = pivot.reindex(columns=EXPECTED_COVERAGES)
        
        data = pivot.values.astype(float)
        mask = np.isnan(data)
        im = ax.imshow(np.ma.array(data, mask=mask), cmap=cmap,
                       vmin=0, vmax=100, aspect="auto")

        # Text values + em-dash for missing + light cell borders for all cells
        for i in range(data.shape[0]):
            for j in range(data.shape[1]):
                # Draw light border around every cell
                ax.add_patch(plt.Rectangle((j-0.5, i-0.5), 1, 1,
                             facecolor="none", edgecolor="#555555", linewidth=0.6))

                # Text on top
                if mask[i, j]:
                    ax.text(j, i, "—", ha="center", va="center",
                            fontsize=9, color="#555555")
                else:
                    val = data[i, j]
                    color = "white" if val < 25 or val > 85 else "black"
                    ax.text(j, i, f"{val:.1f}", ha="center", va="center",
                            fontsize=9, color=color)

        ax.set_xticks(np.arange(len(EXPECTED_COVERAGES)))
        ax.set_xticklabels(EXPECTED_COVERAGES, rotation=0, ha="center", fontsize=12)
        ax.set_yticks(np.arange(len(pivot.index)))
        ax.set_yticklabels(pivot.index.tolist(), fontsize=12)
        ax.set_title(f"{REFERENCE_DISPLAY[ref_id]} — Output Completeness (%)",
                     fontsize=12, pad=8)

    cbar = fig.colorbar(im, ax=axes, orientation="vertical", fraction=0.015, pad=0.02)
    cbar.set_label("Completeness (%)", fontsize=9)

    for ext in ["png", "pdf", "svg"]:
        fig.savefig(FIG_DIR / f"suppfig_S7_completeness_heatmap.{ext}",
                    dpi=300 if ext=="png" else None, bbox_inches="tight")
    plt.close(fig)
    print("✓ completeness_heatmap.png (and pdf/svg) saved with nice tool names")

plot_completeness_heatmap(df_long)

✓ completeness_heatmap.png (and pdf/svg) saved with nice tool names
